# Entrenamiento del Motor Predictivo de Fatiga (SafeShift Predictor)

Este *notebook* evidencia el proceso de ingesta de datos, preparación y entrenamiento del motor de clasificación. 

Dado que el proyecto utiliza datos anonimizados y simulados para cumplir con la Ley 19.628, el modelo se entrenará utilizando el script generador de patrones de distribución reales creado previamente, evitando sesgos operativos. El objetivo de este entrenamiento inicial es superar el umbral del 80% de precisión en la detección de fatiga.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

## 1. Ingesta y Preparación de Datos
Los algoritmos de Machine Learning requieren datos numéricos limpios. En esta etapa, transformamos la respuesta textual ("Sí/No") a un formato binario (0 = Cumple, 1 = Riesgo Crítico) y aislamos exclusivamente las variables operativas (horas de turno acumuladas, descansos compensatorios y turnos de noche consecutivos).

In [ ]:
# 1. Cargar el dataset generado por Javiera
df = pd.read_csv('datos_trabajadores.csv')

# 2. Convertir la variable objetivo a binario (0 = Cumple, 1 = Riesgo/Infracción)
df['Riesgo_Fatiga'] = df['Violacion_Ley_Laboral'].map({'Sí (Infracción)': 1, 'No (Cumple)': 0})

# 3. Seleccionar las variables predictoras (Features)
X = df[['Horas_Semanales', 'Turnos_Noche_Consecutivos', 'Horas_Descanso']]
y = df['Riesgo_Fatiga']

print("Muestra de los datos procesados (X):")
print(X.head())

## 2. Entrenamiento y Evaluación del Modelo
Para validar que el modelo realmente aprende patrones y no solo memoriza el dataset, dividimos los datos: 80% para entrenamiento y 20% para pruebas. Utilizaremos un clasificador de Bosque Aleatorio (`RandomForestClassifier`) dada su robustez con datos tabulares.

In [ ]:
# Dividir en 80% entrenamiento y 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inicializar y entrenar el modelo
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)

# Hacer predicciones con los datos de prueba no vistos
predicciones = modelo_rf.predict(X_test)
precision = accuracy_score(y_test, predicciones)

print(f"--- RESULTADOS DEL MODELO ---")
print(f"Precisión del Modelo: {precision * 100:.2f}%\n")
print("Reporte de Clasificación:")
print(classification_report(y_test, predicciones, target_names=['Bajo Riesgo (0)', 'Riesgo Crítico (1)']))

## 3. Exportación del Modelo
Una vez superada la métrica de precisión, el modelo se empaqueta y exporta. Este archivo `.pkl` será consumido por nuestra API REST en Node.js para evaluar la fatiga de los trabajadores de forma transaccional en el dashboard.

In [ ]:
# Guardar el modelo entrenado para la API
joblib.dump(modelo_rf, 'motor_predictivo_safeshift.pkl')
print("Modelo guardado exitosamente como 'motor_predictivo_safeshift.pkl'")